# Primitive LangGraph

# Getting Charges Data

In [27]:
import os
import sys
from pathlib import Path

import requests
from dotenv import load_dotenv
from langchain.agents import create_agent
from langsmith import Client

import urllib.error
import urllib.request

from langchain.tools import tool
from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model
from IPython.display import Markdown, display
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langchain.tools import tool

from pydantic import BaseModel, Field
import ch_charges as ch_ch
import ch_filing_history as ch_fh
import json
from IPython.display import Image, display

from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated, Literal
import operator
from langgraph.graph import StateGraph, MessagesState, START, END
load_dotenv() # looks for a .env file in the current or parent directions

api_key = os.getenv('ANTHROPIC_API_KEY')
langsmith_api_key = os.getenv('LANGSMITH_API_KEY')

In [28]:
number = "10812571"
since = '2015-01-01'
CATS = ch_fh.SIGNAL_CATEGORIES

filings = ch_fh.get_filing_history(number)
kept    = ch_fh.filter_filings(filings, since=since, categories = CATS)
records = [ch_fh.summarise(f) for f in kept]
charges = ch_ch.get_charges(number)
records_ch = [ch_ch.summarise(c) for c in charges]

In [29]:
records_ch

[{'charge_code': '108125710002',
  'created_on': '2017-07-10',
  'delivered_on': '2017-07-20',
  'satisfied_on': None,
  'status': 'outstanding',
  'classification': 'A registered charge',
  'persons_entitled': ['Interbay Funding Limited'],
  'charge_id': 'fPgLKs6NTqFzZ0_vqa-SMrhO72A'},
 {'charge_code': '108125710001',
  'created_on': '2017-07-10',
  'delivered_on': '2017-07-20',
  'satisfied_on': None,
  'status': 'outstanding',
  'classification': 'A registered charge',
  'persons_entitled': ['Interbay Funding Limited'],
  'charge_id': 'WbYPpoOpnetYtgzrMQrX-zR59RE'}]

## Define tools, model and Structured output

In [37]:
class FilingSummary(BaseModel):
    """What one company's filing history says about its recent activity."""
    headline: str = Field(
        description = "One senctence, 20 words maximum. No lists, no semicolons."    
    )                # one line, human-readable
    key_events: list[str] = Field(
        description="The 2-3 most significant filings. Each must begin with its date.",
        max_length=3,
    )           # the 2-3 filings that matter, each dated
    has_existing_charges: bool = Field(
        description="True if any mortgage-category filing appears in the data provided."
    )        # any category == "mortgage"?
    charge_data: str | None = Field(
        description="If has_existing_charges is true, explain the existing charges data"
    )
    latest_accounts_date: str | None = Field(
        description="The 'made up to' date of the most recent accounts filing, as "
                    "YYYY-MM-DD. This is the accounting period end, NOT the date the "
                    "accounts were filed. Null if no accounts filing is present."
    )
    filing_gaps: str | None = Field(
        description="Late or missing filings worth flagging to a credit analyst. "
                    "Null if the filing record is unremarkable."
    )           # late or missing filings worth flagging
    
model = init_chat_model(
    'claude-sonnet-5',
    temperature = 0
)


## Define State

In [31]:
class ResearchState(TypedDict):
    company_number: str
    filings: list[dict]
    charges: list[dict]
    summary: FilingSummary | None
    messages: Annotated[list[AnyMessage], operator.add]
    
extractor = model.with_structured_output(FilingSummary)

## Specify System prompt

In [32]:
ROLE = """You are a credit analyst reading a UK company's Companies House filing history AND registered chareges, if any.
Summarise what the filings and the charges show about the company's recent activity
and anything relevant to its borrowing position."""

SCOPE = (f"You are seeing only filings in these categories: {', '.join(sorted(CATS))},"
         f"dated {since} or later. Everything else was removed before you saw it. "
         f"Do not draw any conclusion from the absence of other filing types.")

SYSTEM = f"{ROLE}\n\n{SCOPE}"

## Define Nodes

In [33]:
def fetch(state: ResearchState):
    num = state["company_number"]
    kept = ch_fh.filter_filings(ch_fh.get_filing_history(num), since=since, categories=CATS)
    return {
        "filings": [ch_fh.summarise(f) for f in kept],
        "charges": [ch_ch.summarise(c) for c in ch_ch.get_charges(num)],
    }
    
def summarise(state: ResearchState):
    result = extractor.invoke([
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": json.dumps(
            {"filings": state["filings"], "charges": state["charges"]}, indent = 2
        )},
    ])
    return {"summary": result}

## Compile

In [38]:
g = StateGraph(ResearchState)

g.add_node("fetch", fetch)
g.add_node("summarise", summarise)

g.add_edge(START, "fetch")
g.add_edge("fetch", "summarise")
g.add_edge("summarise", END)

agent = g.compile()

In [39]:
result = agent.invoke({"company_number": "10812571"})

In [40]:
print(result['summary'].model_dump_json(indent=2))

{
  "headline": "Active micro-entity with two outstanding charges held by Interbay Funding since incorporation in 2017.",
  "key_events": [
    "2017-06-09: Company incorporated.",
    "2017-07-20: Two registered charges created (charge numbers 108125710001 and 108125710002) in favour of Interbay Funding Limited, both dated 2017-07-10.",
    "2025-12-17: Most recent micro-entity accounts filed, made up to 2024-06-30."
  ],
  "has_existing_charges": true,
  "charge_data": "Two outstanding registered charges (108125710001 and 108125710002), both created on 2017-07-10 and delivered on 2017-07-20, in favour of Interbay Funding Limited. Neither has been satisfied. Interbay Funding is a specialist buy-to-let and commercial mortgage lender, suggesting these charges likely relate to property finance taken out shortly after incorporation. Both charges remain live and would need to be considered by any prospective lender.",
  "latest_accounts_date": "2024-06-30",
  "filing_gaps": "The accounts f